# Run this to enable loading from relative packages

In [ ]:
%load_ext autoreload
%autoreload 2
if "PKG" not in globals():
    root_parent_level = 2
    import importlib, sys, pathlib # https://stackoverflow.com/a/50395128/11996983
    PKG = %pwd
    PKG = pathlib.Path(PKG)
    root = PKG
    full_pkg = f"{root.name}"
    for _ in range(root_parent_level):
        root = root.parent
        full_pkg = f"{root.name}.{full_pkg}"
        MODULE_PATH = str(root / "__init__.py")
        MODULE_NAME = f"{root.name}"
        spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
        module = importlib.util.module_from_spec(spec)
        sys.modules[spec.name] = module
        spec.loader.exec_module(module)
    __package__ = full_pkg


In [ ]:
# This is necessary for interactive plots in Jupyter notebooks
%matplotlib widget

# Save plots with no embedded fonts

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial"]


# Load all mice data

In [ ]:
from .model_runner import loadDF, _extendTrials, _reduceDFSize, DF_FP
from ..behavior.rewardrate import calcAvgRewardRate
# Here we load the behavior dataframe, however we leave in all the trials and
# calculate the loss function only on valid trials. The reason we leave the
# non-valid in is that we need the full history to see the progression of animal
# choices from one trial to the next.
df_behavior = loadDF(df_fp=f"../../{DF_FP}")
print("Calculate average reward rate (and create SessId column)...")
df_behavior = calcAvgRewardRate(df_behavior)
print("Reduce dataframe size to speed up df operations...")
df_behavior = _reduceDFSize(df_behavior)
# We make all the sessions have the same number of trials by extending the
# trials with invalid trials at the end. This is necessary to
# vectorize the model simulation across all sessions.
# The appended trials are marked as invalid and do not contribute to the
# loss function.
print("Extend trial such that all sessions have the same number of trials...")
df_behavior = _extendTrials(df_behavior)

# Load Results

In [ ]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

RESULT_DIR = Path("../../data/RLModel")
FIT_MODES = ("mle", "chisq")

def _load_pickle(path):
    with path.open("rb") as f:
        return pickle.load(f)

def loadFitResults(result_dir=RESULT_DIR):
    fit_results_by_mode = {mode: {} for mode in FIT_MODES}
    all_subjects = set(df_behavior.Name.unique())
    for mode in FIT_MODES:
        for fit_path in sorted(result_dir.glob(f"{mode}_*.pkl")):
            print("Processing:", fit_path)
            loaded = _load_pickle(fit_path)
            fit_results_by_mode[mode][fit_path.name] = loaded
            missing_subjects = all_subjects - set(loaded.keys())
            if missing_subjects:
                print("Missing subjects:", missing_subjects, "from file:", fit_path)
    return fit_results_by_mode

# The structure is:
# {
#   "mle" / "chisq": {
#        "fit_filename.pkl": {
#             "subject_name": subject_fit_result
#        }
#   }
# }
fit_results_by_mode = loadFitResults()


Copy the fit values to the GUI cache such that each subject would be loaded 
with model optimized values 

In [ ]:
import numpy as np

from .model.mle_reeval import parse_fit_filename


def _fit_params_from_result(subject_res):
    assert len(subject_res["params_names"]) == len(subject_res["OptimRes"].x)
    return {
        name: val
        for name, val in zip(subject_res["params_names"], subject_res["OptimRes"].x)
    }


def _fit_loss(fit_type, subject_res):
    if fit_type == "mle":
        return subject_res.get("neg_loglik", np.nan)
    optim_res = subject_res.get("OptimRes")
    return np.nan if optim_res is None else getattr(optim_res, "fun", np.nan)


def _fit_timing(fit_type, subject_res, parsed_t_dur, parsed_dt):
    model_config = subject_res.get("model_config")
    t_dur = subject_res.get("t_dur", None)
    if t_dur is None and model_config is not None:
        t_dur = getattr(model_config, "t_dur", None)
    dt = subject_res.get("dt", None)
    if dt is None and model_config is not None:
        dt = getattr(model_config, "dt", None)
    return (
        parsed_t_dur if t_dur is None else float(t_dur),
        parsed_dt if dt is None else float(dt),
    )


def assignGUISubjectsDefaults(fit_results_by_mode):
    """Nest every loaded fit into the {t_dur: {noise: {bias: {drift: {subject:
    {fit_key: ...}}}}}} shape ``visualize.createWidget`` looks fits up in.

    Filenames are decomposed by ``model/mle_reeval.py:parse_fit_filename``
    (unit-tested, and the same parser ``rlmodel_compare.ipynb`` uses), so the
    two notebooks stay in sync when ``fit.evolveFP`` grows a suffix or a
    drift family.

    Two keying details matter:

    * The DRIFT axis uses ``fid.drift`` -- the internal ``DRIFT_FN_DICT``
      key (``NoiseGain-`` / ``Bound-`` / ``DriftGain-RewardRate``...), NOT
      ``fid.drift_alias``. That is what ``visualize._resolved_drift_fn_value``
      resolves the Drift Fn dropdown + Scale-How + "RR as Drift" controls
      down to, so the reward-rate channels stay distinguishable here.
      (``drift_alias`` is the display collapse used by rlmodel_compare.)
    * ``fit_key`` appends ``fid.variant_suffix`` (scaledB, then joint
      weights -- evolveFP order) so a plain, a scale-bound and a
      joint pickle for the same (subject, model, mode) don't collide. It must
      match ``visualize._variant_suffix``, which builds the same string from
      the GUI widget state.
    """
    subjects_defaults = {}

    for fit_type, fit_results in fit_results_by_mode.items():
        for filename, fit_res_dict in fit_results.items():
            print("Processing:", filename)
            fid = parse_fit_filename(filename)
            fit_key = f"{fit_type}{fid.variant_suffix}"

            for subject, subject_res in fit_res_dict.items():
                t_dur, dt = _fit_timing(fit_type, subject_res,
                                        fid.t_dur, fid.dt)
                cur_subject_defaults = _fit_params_from_result(subject_res)

                subject_entry = subjects_defaults.setdefault(t_dur, {}) \
                    .setdefault(fid.noise, {}) \
                    .setdefault(fid.bias, {}) \
                    .setdefault(fid.drift, {}) \
                    .setdefault(subject, {})
                assert fit_key not in subject_entry, (
                    f"Subject {subject} from {filename} already has {fit_key} "
                    f"defaults for {t_dur}s - {fid.noise} {fid.bias} {fid.drift}")
                subject_entry[fit_key] = {
                    "params": cur_subject_defaults,
                    "result": subject_res,
                    "loss": _fit_loss(fit_type, subject_res),
                    "dt": dt,
                    "t_dur": t_dur,
                    "filename": filename,
                }

    return subjects_defaults


subjects_defaults = assignGUISubjectsDefaults(fit_results_by_mode=fit_results_by_mode)


In [ ]:
# fit_results_by_mode["mle"]
# fit_results_by_mode["chisq"]


In [ ]:
def assignSessId(df):
    df = df.copy()
    df["SessId"] = df.apply(
            lambda x: f"{x['Name']}_{x['Date']}_{x['SessionNum']}",
            axis=1)
    df["DVabs"] = df.DV.abs()
    df["Q_val"] = np.nan
    df["Q_L"] = np.nan
    df["Q_R"] = np.nan
    df["RewardRate"] = np.nan
    return df
all_df = assignSessId(df_behavior)

# Interactive Viewer

The model is picked with four controls in the first column:

* **Drift Fn** — `RewardRate*` is an alias; the three controls below choose
  which implementation it resolves to.
* **Scale-How** — which of (`NOISE_SIGMA`, `BOUND`) is the fitted scale axis.
  `Bound` corresponds to `--scale-bound` / the `_scaledB` fits.
* **RR as Drift** — send the learned reward rate `r` to the **drift**
  (`d += DV·V·g(r)·dt`, noise and threshold flat) instead of the noise or the
  threshold. This *overrides* the other channel rather than adding to it;
  the same switch is `--use-drift-rr` on `model_runner.py`.
* **RR-Drift Map** — `g(r)`: `2-r` (default; a high reward rate *weakens* the
  drift) or `1+r` (a high reward rate strengthens it, matching the speed
  direction of the noise/threshold channels).

Hover any of them for the full explanation. Together they resolve to one
`DRIFT_FN_DICT` key — `NoiseGain-` / `Bound-` / `DriftGain-` /
`DriftGain(1+r)-RewardRate` — and that key is part of the saved-fit filename,
so switching a control loads that variant's own fit (or leaves the sliders
alone and shows "not run" when it hasn't been fit).


In [ ]:
import pickle
import pathlib
# This is a very simple helper that remembers the last GUI setting used. This is
# useful when you want to share settings between runs or between code users.
# Remember to save the cache after changing settings!
GUI_CACHE_FP = pathlib.Path("model_GUI_cache.pkl")
if GUI_CACHE_FP.exists():
    print("Loading GUI cache...")
    with GUI_CACHE_FP.open("rb") as fp_pp:
        gui_cache = pickle.load(fp_pp)
else:
    print("No GUI cache found, starting fresh...")
    gui_cache = {}

In [ ]:
gui_cache = {}

In [ ]:
from .model.initvals import InitVals, DT, T_dur
init_vals = InitVals()

In [ ]:
# %prun -s cumulative
from .model import visualize
visualize.createWidget(init_vals=init_vals, gui_cache=gui_cache,
                       df=all_df,
                       dt=DT, t_dur=T_dur,
                       subjects_defaults=subjects_defaults,
                       save_figs=False,
                       is_small_fig_mode=False,
                       )


In [ ]:
raise error

In [ ]:
with GUI_CACHE_FP.open("wb") as fp_pp:
    pickle.dump(gui_cache, fp_pp)